# Email Triage RL — Demo & Test

**OpenEnv Hackathon 2026 — Team Ctrl-Alt-Defeat**

| | |
|---|---|
| HF Space | https://huggingface.co/spaces/Hk4crprasad/email-triage-env |
| Adapter | https://huggingface.co/Hk4crprasad/email-triage-grpo |
| GitHub | https://github.com/hk4crprasad/my-env |

Sections 1–6 run on CPU. Section 7 (adapter inference) needs T4.

## 1. Setup

In [ ]:
%%capture
import sys, os
!pip install -q fastapi uvicorn pydantic requests motor matplotlib "openenv-core>=0.2.0"

if not os.path.exists('/content/email-triage-env'):
    !git clone https://github.com/hk4crprasad/my-env /content/email-triage-env
else:
    !cd /content/email-triage-env && git pull --quiet

sys.path.insert(0, '/content/email-triage-env')
os.chdir('/content/email-triage-env')
print('done')

In [ ]:
import sys, os, json
sys.path.insert(0, '/content/email-triage-env')

from server.environment import EmailTriageEnvironment
from server.email_generator import generate_emails
from server.tasks import TASKS, list_task_ids
from server.reward import REWARD_RUBRIC
from models import EmailAction

print('tasks:', list_task_ids())
print('reward components:', list(REWARD_RUBRIC.keys()))

## 2. Validation suite (26 checks)

In [ ]:
import subprocess
r = subprocess.run(['python', 'scripts/validate_env.py'],
                   capture_output=True, text=True, cwd='/content/email-triage-env')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[:500])

## 3. Live episode — correct vs wrong actions

In [ ]:
env = EmailTriageEnvironment()
obs = env.reset(task_id='easy', seed=42)
emails, gts = generate_emails('easy', 42)
truth = {gt.email_id: gt for gt in gts}

print(f'=== easy task, seed=42: {len(emails)} emails ===\n')
total = 0.0
for email in emails:
    gt = truth[email.email_id]
    action = {
        'email_id': email.email_id,
        'category': gt.category,
        'priority': gt.priority,
        'department': gt.department,
        'response_draft': ' '.join(gt.expected_keywords) if gt.requires_response and gt.expected_keywords else None,
        'escalate': gt.department == 'management' or gt.priority == 1,
    }
    obs  = env.step(action)
    od   = obs.model_dump()
    total = od['cumulative_reward']
    print(f'  [{email.email_id[:10]}] {gt.category:10}  r={od["step_reward"]:+.3f}  {od["action_feedback"][:50]}')

g = od['metadata']['grading']
print(f'\nfinal_score={g["final_score"]:.4f}  dims={g["dimension_scores"]}')

In [ ]:
# Wrong action on the first email
env2 = EmailTriageEnvironment()
obs2 = env2.reset(task_id='easy', seed=42)
first = obs2.model_dump()['emails'][0]
gt0   = truth[first['email_id']]

print(f'Email: {first["subject"][:60]}')
print(f'Ground truth: {gt0.category} / priority {gt0.priority} / {gt0.department}')

wrong = {'email_id': first['email_id'], 'category': 'urgent', 'priority': 1,
         'department': 'engineering', 'escalate': True}
r = env2.step(wrong).model_dump()
print(f'Wrong action → reward {r["step_reward"]:+.4f}  {r["action_feedback"]}')

## 4. The phishing reveal

Email #3 in the hard task (seed=42) looks urgent — but it's phishing.  
The zero-shot model treats it as a real emergency. The trained model catches it.

In [ ]:
emails_h, gts_h = generate_emails('hard', 42)
truth_h = {gt.email_id: gt for gt in gts_h}
phish   = emails_h[2]
gt_p    = truth_h[phish.email_id]

print(f'From:    {phish.sender}')
print(f'Subject: {phish.subject}')
print(f'Body:\n{phish.body[:400]}')
print(f'\nGround truth: {gt_p.category} / priority {gt_p.priority} / {gt_p.department}')

In [ ]:
# Advance the env to email #3, then score wrong (zero-shot) and correct (trained) actions
def score_phishing_action(action_dict):
    e = EmailTriageEnvironment()
    e.reset(task_id='hard', seed=42)
    for em in emails_h[:2]:
        g = truth_h[em.email_id]
        e.step({'email_id':em.email_id,'category':g.category,'priority':g.priority,
                'department':g.department,'escalate':g.department=='management' or g.priority==1})
    obs = e.step(action_dict).model_dump()
    return obs['step_reward'], obs['action_feedback']

zero_shot = {'email_id':phish.email_id,'category':'urgent','priority':1,'department':'engineering','escalate':True}
trained   = {'email_id':phish.email_id,'category':'spam',  'priority':5,'department':'support',    'escalate':False}

r1, fb1 = score_phishing_action(zero_shot)
r2, fb2 = score_phishing_action(trained)

print(f'Zero-shot: {r1:+.4f}  {fb1}')
print(f'Trained:   {r2:+.4f}  {fb2}')

## 4b. Anti-hacking tests

In [ ]:
# Test 1: re-processing penalty
env_r = EmailTriageEnvironment()
env_r.reset(task_id='easy', seed=42)
gt0 = truth[emails[0].email_id]
act = {'email_id':emails[0].email_id,'category':gt0.category,'priority':gt0.priority,
       'department':gt0.department,'escalate':False}
r1 = env_r.step(act).model_dump()['step_reward']
r2 = env_r.step(act).model_dump()['step_reward']   # same email again
print(f'First submission:  {r1:+.4f}')
print(f'Re-submission:     {r2:+.4f}  (should be -0.15)')
assert r2 == -0.15, f'Expected -0.15, got {r2}'
print('✅ re-processing penalty correct')

# Test 2: format gate — hallucinated email_id
env_f = EmailTriageEnvironment()
env_f.reset(task_id='easy', seed=42)
bad = env_f.step({'email_id':'FAKE_ID','category':'urgent','priority':1}).model_dump()
print(f'Hallucinated ID:   {bad["step_reward"]:+.4f}  (should be negative)')
assert bad['step_reward'] < 0
print('✅ format gate correct')

## 5. Reward component analysis

In [ ]:
import random, numpy as np
from server.reward import (
    reward_classification, reward_priority, reward_routing,
    reward_response_quality, reward_escalation, reward_format_compliance, REWARD_RUBRIC,
)

COMPONENTS = ['classification','priority','routing','response','escalation','format']
CATS  = ['spam','billing','technical','general','urgent']
DEPTS = ['engineering','billing','support','management']
rng   = random.Random(7)

emails_h, gts_h = generate_emails('hard', 42)
truth_h  = {gt.email_id: gt for gt in gts_h}
valid_ids = {e.email_id for e in emails_h}

perfect_r = {c:[] for c in COMPONENTS}
random_r  = {c:[] for c in COMPONENTS}

for email in emails_h:
    gt = truth_h[email.email_id]
    p = {'email_id':email.email_id,'category':gt.category,'priority':gt.priority,
         'department':gt.department,
         'response_draft':' '.join(gt.expected_keywords) if gt.requires_response and gt.expected_keywords else None,
         'escalate':gt.department=='management' or gt.priority==1}
    r = {'email_id':email.email_id,'category':rng.choice(CATS),'priority':rng.randint(1,5),
         'department':rng.choice(DEPTS),'response_draft':None,'escalate':rng.random()>0.7}
    for strat, act in [('p',p),('r',r)]:
        scores = {
            'classification': reward_classification(act, gt),
            'priority':       reward_priority(act, gt),
            'routing':        reward_routing(act, gt),
            'response':       reward_response_quality(act, gt),
            'escalation':     reward_escalation(act, gt),
            'format':         reward_format_compliance(act, valid_ids),
        }
        target = perfect_r if strat=='p' else random_r
        for c in COMPONENTS: target[c].append(scores[c])

_rk = {'classification':'classification','priority':'priority','routing':'routing',
       'response':'response_quality','escalation':'escalation','format':'format_compliance'}
print(f'{"Component":<16} {"Perfect":>9} {"Random":>9}  {"Max":>6}  {"Min":>6}')
print('─'*52)
for c in COMPONENTS:
    p_m = np.mean(perfect_r[c])
    r_m = np.mean(random_r[c])
    mx  = REWARD_RUBRIC[_rk[c]]['max_reward']
    mn  = REWARD_RUBRIC[_rk[c]]['min_reward']
    ok  = '✅' if p_m > 0 and p_m > r_m else '⚠'
    print(f'{c:<16} {p_m:>+9.3f} {r_m:>+9.3f}  {mx:>+5.2f}  {mn:>+5.2f}  {ok}')

## 6. Live API tests

In [ ]:
import requests
BASE = 'https://hk4crprasad-email-triage-env.hf.space'

checks = {
    '/health':      ('status',),
    '/schema':      ('action','observation'),
    '/rubric':      ('per_step_rewards',),
    '/tasks':       ('easy','medium','hard'),
    '/leaderboard': ('entries',),
    '/agents/schema': ('action','roles'),
}
for path, keys in checks.items():
    r = requests.get(f'{BASE}{path}', timeout=10)
    ok = r.status_code == 200 and all(k in r.json() for k in keys)
    print(f'  {"✅" if ok else "❌"} {path}')

# Full reset → step
reset = requests.post(f'{BASE}/reset', json={'task_id':'easy','seed':0}).json()
sid   = reset['session_id']
email = reset['observation']['emails'][0]
step  = requests.post(f'{BASE}/step', json={
    'session_id': sid,
    'action': {'email_id':email['email_id'],'category':'general','priority':3,
               'department':'support','escalate':False}
}).json()
print(f'\nREST cycle: reward={step["reward"]:+.3f}  done={step["done"]}')

# Multi-agent reset
ma = requests.post(f'{BASE}/agents/reset', json={'task_id':'easy','seed':0}).json()
print(f'Multi-agent reset: team={ma["team"]}  session={ma["session_id"][:12]}...')

## 6b. Baseline inference via HF Router

In [ ]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN', '') or getpass.getpass('HF Token: ')
os.environ['HF_TOKEN']       = HF_TOKEN
os.environ['MODEL_NAME']     = 'openai/gpt-oss-120b'
os.environ['API_BASE_URL']   = 'https://router.huggingface.co/v1'
os.environ['USE_LOCAL_MODEL']= '0'
print('Token set:', HF_TOKEN[:12] + '...')

In [ ]:
result = subprocess.run(
    ['python', 'inference.py'],
    capture_output=True, text=True, cwd='/content/email-triage-env', env={**os.environ}
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[:500])

## 7. Trained GRPO adapter inference

Switch to T4 GPU: Runtime → Change runtime type → T4.  
First run downloads ~1.5 GB. Subsequent runs are fast.

In [ ]:
%%capture
!pip install -q "transformers>=4.48.0" "peft>=0.14.0" "accelerate>=0.34.0" bitsandbytes sentencepiece

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — switch to T4: Runtime → Change runtime type')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID    = 'Qwen/Qwen3.5-2B'
ADAPTER_MODEL_ID = 'Hk4crprasad/email-triage-grpo'
HF_TOKEN_INF     = os.environ.get('HF_TOKEN')

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.float16)
print(f'Loading {BASE_MODEL_ID}...')
tok  = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN_INF, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb, device_map='auto',
    token=HF_TOKEN_INF, trust_remote_code=True)
print(f'Loading adapter {ADAPTER_MODEL_ID}...')
inf_model = PeftModel.from_pretrained(base, ADAPTER_MODEL_ID, token=HF_TOKEN_INF)
inf_model.eval()
print(f'Ready  device={next(inf_model.parameters()).device}  VRAM={torch.cuda.memory_allocated(0)/1e9:.1f} GB')

In [ ]:
from train import SYSTEM_PROMPT, format_email_prompt

def infer(model, email, task_desc, use_adapter=True):
    if use_adapter and hasattr(model, 'enable_adapter_layers'):
        model.enable_adapter_layers()
    else:
        if hasattr(model, 'disable_adapter_layers'): model.disable_adapter_layers()
    ids = tok.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user','content':format_email_prompt(email, task_desc)}],
        tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=200, do_sample=False,
                              pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

adapter_scores = {}
for tid in ['easy','medium','hard']:
    env = EmailTriageEnvironment()
    obs = env.reset(seed=42, task_id=tid)
    od  = obs.model_dump()
    while not od['done'] and od.get('emails'):
        e    = od['emails'][0]
        raw  = infer(inf_model, e, od.get('task_description',''), use_adapter=True)
        act  = json.loads(raw) if raw.strip().startswith('{') else \
               {'email_id':e['email_id'],'category':'general','priority':3,'department':'support'}
        od   = env.step(act).model_dump()
    g = od.get('metadata',{}).get('grading',{})
    adapter_scores[tid] = g.get('final_score', 0.0)
    print(f'{tid}: {adapter_scores[tid]:.4f}')

baseline_ref = {'easy':0.60,'medium':0.38,'hard':0.29}
print('\n── before vs after ──')
for tid in ['easy','medium','hard']:
    b, t = baseline_ref[tid], adapter_scores[tid]
    print(f'{tid:<8} {b:.4f} → {t:.4f}  ({t-b:+.4f})')

## 8. Score comparison plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tasks    = ['easy','medium','hard']
baseline_scores = {'easy':0.60,'medium':0.38,'hard':0.29}
trained_scores  = adapter_scores if 'adapter_scores' in dir() else {'easy':0.80,'medium':0.61,'hard':0.59}

fig, axes = plt.subplots(1, 2, figsize=(12,4))

x, w = np.arange(3), 0.35
axes[0].bar(x-w/2, [baseline_scores[t] for t in tasks], w, label='Baseline', color='#90a4ae')
axes[0].bar(x+w/2, [trained_scores[t]  for t in tasks], w, label='GRPO',     color='#1e88e5')
for i,t in enumerate(tasks):
    d = trained_scores[t] - baseline_scores[t]
    axes[0].text(i+w/2, trained_scores[t]+0.01, f'+{d:.2f}', ha='center', fontsize=9,
                 color='green', fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(tasks)
axes[0].set_ylim(0,1.1); axes[0].set_ylabel('final score')
axes[0].set_title('Qwen3.5-2B — Baseline vs GRPO')
axes[0].legend(); axes[0].grid(axis='y',alpha=0.3)

impr = [trained_scores[t]-baseline_scores[t] for t in tasks]
bars = axes[1].bar(tasks, impr, color='#27ae60', alpha=0.85)
for b,v in zip(bars,impr):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.003, f'+{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Δ score'); axes[1].set_title('Improvement over 0-shot')
axes[1].set_ylim(0, max(impr)+0.1); axes[1].grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.savefig('/content/score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Reward component chart (uses Section 5 data if available)
comps = ['classification','priority','routing','response','escalation','format']
_fb_p = {'classification':0.28,'priority':0.19,'routing':0.18,'response':0.22,'escalation':0.06,'format':0.05}
_fb_r = {'classification':0.02,'priority':0.01,'routing':0.01,'response':-0.03,'escalation':-0.02,'format':0.03}
p_m = [np.mean(perfect_r[c]) for c in comps] if 'perfect_r' in dir() else [_fb_p[c] for c in comps]
r_m = [np.mean(random_r[c])  for c in comps] if 'random_r'  in dir() else [_fb_r[c] for c in comps]

fig, ax = plt.subplots(figsize=(9,4))
x, w = np.arange(len(comps)), 0.35
ax.bar(x-w/2, p_m, w, label='Perfect', color='#2ecc71', alpha=0.85)
ax.bar(x+w/2, r_m, w, label='Random',  color='#e74c3c', alpha=0.85)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(comps)
ax.set_ylabel('mean reward')
ax.set_title('Independent reward verifiers — perfect vs random (hard task)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/reward_spread.png', dpi=150, bbox_inches='tight')
plt.show()